# 01 - Encode trajectories on a Kaggle GPU (Day 4)
Settings: **Accelerator GPU T4**, **Internet on**. Add your HF token as a Kaggle *Secret* named `HF_TOKEN`.

Inputs: a private Kaggle Dataset `tw-data` containing the `data/processed` folder produced on your laptop by
`tracewarden data agentdrift ...` (train/val/test/train_shuf .jsonl).

In [ ]:
!pip -q install "git+https://github.com/<you>/tracewarden.git#egg=tracewarden[st]" huggingface_hub
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
from tracewarden.encoders import SentenceTransformerEncoder, encode_corpus
from tracewarden.schema import load_jsonl
import glob

trajs, seen = [], set()
for f in sorted(glob.glob('/kaggle/input/tw-data/processed/*.jsonl')):
    for t in load_jsonl(f):
        if t.id not in seen:
            seen.add(t.id); trajs.append(t)
print(len(trajs), 'trajectories', sum(len(t.steps) for t in trajs), 'steps')

In [ ]:
# main encoder (Days 5-17)
enc = SentenceTransformerEncoder('sentence-transformers/all-mpnet-base-v2', device='cuda', batch_size=256)
encode_corpus(trajs, enc, '/kaggle/working/cache/mpnet')

# fast mode (Day 19) and multilingual encoder (Day 18) - same data, different caches
encode_corpus(trajs, SentenceTransformerEncoder('sentence-transformers/all-MiniLM-L6-v2', device='cuda', batch_size=512),
              '/kaggle/working/cache/minilm')
encode_corpus(trajs, SentenceTransformerEncoder('sentence-transformers/paraphrase-multilingual-mpnet-base-v2', device='cuda', batch_size=256),
              '/kaggle/working/cache/multilingual')

In [ ]:
# push to a private HF dataset repo; pull on the laptop with snapshot_download
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login
login(UserSecretsClient().get_secret('HF_TOKEN'))
api = HfApi(); repo = '<you>/tracewarden-cache'
api.create_repo(repo, repo_type='dataset', private=True, exist_ok=True)
api.upload_folder(folder_path='/kaggle/working/cache', repo_id=repo, repo_type='dataset')

On the laptop:
```python
from huggingface_hub import snapshot_download
snapshot_download('<you>/tracewarden-cache', repo_type='dataset', local_dir='cache')
```